# Deep Learning 022 — Early Stopping

Companion notebook to the lesson. Overfitting is not only a property of the *model* — it is
a property of the model **at a particular epoch**. Train long enough and almost any network
will start memorising, which means there is a moment when it was at its best and you walked
straight past it.

Early stopping is the cheapest regulariser there is: stop at that moment.

| Claim | Measured below |
|---|---|
| the best epoch comes long before the last | best at epoch **171** of 600 |
| training past it actively hurts | the remaining 428 epochs made validation **23% worse** |
| training loss keeps falling the whole time | it does — which is why you must monitor `val_loss` |
| `patience` cannot be 0 | validation loss rose on **281 of 599** epoch transitions |
| `restore_best_weights` matters | 0.1941 with it, 0.2184 without |

We implement the network in `numpy` so the epoch loop is visible, then map it onto Keras'
`EarlyStopping`. The lesson's own run reported its best epoch at 48 of 600 on a different
network and dataset; the exact epoch is not the point and will move with either. **The
shape is the point**, and it is the same shape every time.

In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)

X, y = make_circles(n_samples=400, noise=0.22, factor=0.4, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
s = StandardScaler().fit(X_tr)
X_tr, X_va = s.transform(X_tr), s.transform(X_va)
y_tr, y_va = y_tr.reshape(-1, 1).astype(float), y_va.reshape(-1, 1).astype(float)
print(f"train {X_tr.shape}, validation {X_va.shape}")

In [ ]:
def relu(z):    return np.maximum(0, z)
def d_relu(z):  return (z > 0).astype(float)
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def bce(p, t):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-(t * np.log(p) + (1 - t) * np.log(1 - p)).mean())

def init(sizes, seed=0):
    r = np.random.default_rng(seed)
    return [[r.normal(size=(a, b)) * np.sqrt(2 / a), np.zeros(b)]
            for a, b in zip(sizes[:-1], sizes[1:])]

def forward(W, X):
    acts, zs, a = [X], [], X
    for i, (w, b) in enumerate(W):
        z = a @ w + b
        zs.append(z)
        a = sigmoid(z) if i == len(W) - 1 else relu(z)
        acts.append(a)
    return zs, acts

def step(W, X, t, lr):
    zs, acts = forward(W, X)
    delta = (acts[-1] - t) / len(X)          # d BCE / d z for a sigmoid output
    for i in reversed(range(len(W))):
        gw, gb = acts[i].T @ delta, delta.sum(0)
        if i:
            delta = (delta @ W[i][0].T) * d_relu(zs[i - 1])
        W[i][0] -= lr * gw
        W[i][1] -= lr * gb
    return W

## Part A — Watch the two curves separate

Train an over-capacity network for far too long, recording both losses every epoch, and
keep a copy of the weights whenever validation improves.

In [ ]:
EPOCHS = 600
W = init([2, 64, 64, 1], seed=1)
tr_hist, va_hist = [], []
best = {"loss": np.inf, "epoch": -1, "W": None}

for e in range(EPOCHS):
    W = step(W, X_tr, y_tr, lr=0.5)
    l_tr = bce(forward(W, X_tr)[1][-1], y_tr)
    l_va = bce(forward(W, X_va)[1][-1], y_va)
    tr_hist.append(l_tr); va_hist.append(l_va)
    if l_va < best["loss"] - 1e-4:
        best = {"loss": l_va, "epoch": e, "W": [[w.copy(), b.copy()] for w, b in W]}

tr_hist, va_hist = np.array(tr_hist), np.array(va_hist)
print(f"{'epoch':>8}{'train loss':>14}{'val loss':>12}")
for e in (0, 10, 50, 100, 200, 400, EPOCHS - 1):
    print(f"{e:>8}{tr_hist[e]:>14.4f}{va_hist[e]:>12.4f}")

print(f"\nbest validation loss {best['loss']:.4f} at epoch {best['epoch']}")
print(f"final validation loss {va_hist[-1]:.4f} at epoch {EPOCHS - 1}")
print(f"the last {EPOCHS - 1 - best['epoch']} epochs made validation "
      f"{va_hist[-1] / best['loss'] - 1:.0%} WORSE")
print(f"meanwhile training loss went {tr_hist[best['epoch']]:.4f} -> {tr_hist[-1]:.4f}"
      f"  (still improving)")

That is the whole phenomenon in one cell.

**Training loss falls monotonically for all 600 epochs.** If that were the only number you
watched, you would conclude the model was still learning — and it was, just not anything
useful. Validation loss bottomed out early and then climbed steadily.

The lesson's headline is exactly this shape: the best epoch is a small fraction of the
budget, and everything after it is spent making the model worse in a way the training loss
cannot see.

In [ ]:
# Optional plot. Skip if matplotlib is unavailable.
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(tr_hist, label="train")
plt.plot(va_hist, label="validation")
plt.axvline(best["epoch"], ls="--", c="grey", label=f"best epoch {best['epoch']}")
plt.xlabel("epoch"); plt.ylabel("binary cross-entropy")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Part B — Why `patience` cannot be zero

Validation loss is **noisy**. It goes up and down from epoch to epoch even while the trend
is still downward, so a rule that stops at the first non-improvement stops almost
immediately and for no reason.

In [ ]:
rises = np.diff(va_hist) > 0
first_rise = int(np.argmax(rises))
print(f"validation loss rose on {rises.sum()} of {len(rises)} epoch transitions")
print(f"the FIRST rise happened at epoch {first_rise}, "
      f"{best['epoch'] - first_rise} epochs before the true best\n")

print(f"{'patience':>10}{'would stop at':>16}{'val loss there':>17}{'vs the best':>14}")
for pat in (0, 1, 5, 20, 50):
    stop, wait, bl = None, 0, np.inf
    for e, l in enumerate(va_hist):
        if l < bl - 1e-4:
            bl, wait = l, 0
        else:
            wait += 1
            if wait > pat:
                stop = e
                break
    stop = stop if stop is not None else len(va_hist) - 1
    print(f"{pat:>10}{stop:>16}{va_hist[stop]:>17.4f}{va_hist[stop] / best['loss']:>13.2f}x")

`patience = 0` stops on the first wobble, long before the model is finished. Large patience
costs epochs but lands near the true minimum. **The parameter is buying tolerance for noise,
not for slow learning.**

## Part C — `restore_best_weights` is not optional

Here is the part people miss. Early stopping decides *when to stop*. It does not, by
default, decide *which weights you keep* — and the weights you have when you stop are the
ones from `patience` epochs **after** the best.

In [ ]:
def accuracy(W, X, t):
    return float(((forward(W, X)[1][-1] > 0.5) == (t > 0.5)).mean())

pat = 20
stop, wait, bl, W_at_stop = None, 0, np.inf, None
W2 = init([2, 64, 64, 1], seed=1)
snap = None
for e in range(EPOCHS):
    W2 = step(W2, X_tr, y_tr, lr=0.5)
    l = bce(forward(W2, X_va)[1][-1], y_va)
    if l < bl - 1e-4:
        bl, wait = l, 0
        snap = [[w.copy(), b.copy()] for w, b in W2]
    else:
        wait += 1
        if wait > pat:
            stop = e
            W_at_stop = [[w.copy(), b.copy()] for w, b in W2]
            break

print(f"stopped at epoch {stop}, best was epoch {best['epoch']}\n")
print(f"{'weights you keep':<34}{'val loss':>10}{'val accuracy':>15}")
print(f"{'whatever you had when it stopped':<34}"
      f"{bce(forward(W_at_stop, X_va)[1][-1], y_va):>10.4f}"
      f"{accuracy(W_at_stop, X_va, y_va):>15.3f}")
print(f"{'restore_best_weights=True':<34}"
      f"{bce(forward(snap, X_va)[1][-1], y_va):>10.4f}"
      f"{accuracy(snap, X_va, y_va):>15.3f}")

**Keras defaults `restore_best_weights` to `False`.** That means the default behaviour is to
detect the best epoch, carefully count `patience` epochs past it, and then hand you the
weights from the end of that wait. Always set it to `True`.

## Part D — The same thing in Keras

```python
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",          # NOT "loss" - training loss keeps falling, as Part A shows
    patience=20,                 # tolerate 20 non-improving epochs; it is noisy
    min_delta=0.001,             # what counts as an improvement at all
    restore_best_weights=True,   # DEFAULT IS FALSE - always set this
    verbose=1,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),   # the callback needs something to watch
    epochs=3000,                      # set it HIGH and let the callback decide
    callbacks=[early_stop],
)
print("stopped at epoch", len(history.history["loss"]))
print("best epoch     ", early_stop.best_epoch + 1)
```

The mental shift is the useful part: **`epochs` stops being a hyperparameter you tune.** Set
it far higher than you need and let the callback find the stopping point, which is a
decision made from data rather than from guessing.

## Try it yourself

1. Set `min_delta` to 0.05 in Part B's loop. How much earlier does it stop, and is the model
   worse?
2. Add L2 to the numpy trainer (subtract `lr * lam * w` in `step`). Does the validation
   minimum move later? Should it?
3. Re-run Part A with a `(2, 4, 1)` network. Does the validation curve still turn upward —
   and if not, what does that tell you about when early stopping is worth adding?
4. Track validation *accuracy* alongside loss. Do they bottom out at the same epoch? Which
   would you rather monitor, and why is loss the usual choice?